# ERP Stok Kartı Tekilleştirme - İlk Veri Analizi

Bu notebook, ERP sisteminden alınan anonim stok kartı verisi üzerinde ilk veri inceleme, temizleme ve benzerlik analizi adımlarını içerir.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Veri yolu - GitHub reposunda dosyayı data klasörüne koy
DATA_PATH = '../data/data.xlsx'

df = pd.read_excel(DATA_PATH)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/dataSON.xlsx'

In [ ]:
# Veri seti genel bilgileri
print('Satır / Sütun:', df.shape)
print('\nKolonlar:')
print(df.columns.tolist())

print('\nEksik veri sayıları:')
print(df.isnull().sum())

In [ ]:
# Temel temizlik
df['STOK ADI'] = df['STOK ADI'].astype(str).str.strip().str.upper()
df['YALIN HALİ'] = df['YALIN HALİ'].astype(str).str.strip().str.upper()
df['STOK KODU'] = df['STOK KODU'].astype(str).str.strip().str.upper()

df.head()

In [ ]:
# Aynı stok adına sahip kayıtları bulma
duplicate_names = df[df.duplicated(subset=['STOK ADI'], keep=False)].sort_values('STOK ADI')
duplicate_names[['STOK KODU', 'STOK ADI', 'YALIN HALİ']].head(20)

In [ ]:
# Stok adı bazlı TF-IDF + Cosine Similarity
# Amaç: yazımı benzer olan stok kartlarını skorlayarak tespit etmek

texts = df['STOK ADI'].fillna('').astype(str)

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
tfidf_matrix = vectorizer.fit_transform(texts)

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_matrix.shape

In [ ]:
# Belirli bir stok kaydı için en benzer kayıtları listeleme

def get_similar_items(index, top_n=10):
    scores = list(enumerate(similarity_matrix[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != index]
    top_scores = scores[:top_n]
    
    results = []
    for item_index, score in top_scores:
        results.append({
            'ana_stok_kodu': df.iloc[index]['STOK KODU'],
            'ana_stok_adi': df.iloc[index]['STOK ADI'],
            'benzer_stok_kodu': df.iloc[item_index]['STOK KODU'],
            'benzer_stok_adi': df.iloc[item_index]['STOK ADI'],
            'benzerlik_skoru': round(score, 4)
        })
    return pd.DataFrame(results)

get_similar_items(0, top_n=10)

In [ ]:
# Tüm veri için belirli eşik üzerindeki benzer kayıt çiftlerini bulma

threshold = 0.85
pairs = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        score = similarity_matrix[i, j]
        if score >= threshold:
            pairs.append({
                'stok_kodu_1': df.iloc[i]['STOK KODU'],
                'stok_adi_1': df.iloc[i]['STOK ADI'],
                'stok_kodu_2': df.iloc[j]['STOK KODU'],
                'stok_adi_2': df.iloc[j]['STOK ADI'],
                'benzerlik_skoru': round(score, 4)
            })

similar_pairs = pd.DataFrame(pairs)
similar_pairs.head(20)

In [ ]:
# Sonuçları CSV olarak kaydetme
similar_pairs.to_csv('../data/benzer_stok_kartlari.csv', index=False, encoding='utf-8-sig')

print('Benzer stok kartı çifti sayısı:', len(similar_pairs))